# Generative AI 003 — Why LangChain Is Needed

The one-line definition — "an open-source framework for developing applications powered by
LLMs" — is accurate and tells you nothing about the problem it solves. So this notebook builds
the interesting part of a real application and lets the need appear on its own.

| Part | What we measure |
|---|---|
| A | keyword search: precision **0.40**, and recall **0.00** for the key word alone |
| B | the retrieval pipeline — embed, compare, take the top k |
| C | where it breaks: a paraphrase scores **exactly 0.0000** against everything |
| D | why retrieve at all — the token arithmetic |

Needs `scikit-learn`. No API key, no downloads.

In [ ]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Part A — Why keyword search is not good enough

A small stand-in for a machine-learning textbook, one entry per page. Pages **372** and **461**
are the two that actually answer "what are the assumptions of linear regression?".

Read page 461 carefully before you run the search.

In [ ]:
TEXTBOOK = {
    12:  "Assumptions are important in statistics. This page assumes the reader knows algebra.",
    88:  "We assume the data is clean before training any model on it.",
    150: "Decision trees make no assumptions about the distribution of the data.",
    372: "Linear regression relies on linearity, independence of errors, homoscedasticity "
         "and normally distributed residuals.",
    404: "The assumptions behind hypothesis testing are covered in the appendix.",
    461: "Before fitting a linear model, check that the residuals have constant variance "
         "and that the predictors are not strongly collinear.",
    503: "Neural networks assume very little about the input, which is part of their appeal.",
}
RELEVANT = {372, 461}

def keyword_search(words):
    '''Whole-word match, so "linear" does not match inside "collinear".'''
    hits = []
    for page, text in TEXTBOOK.items():
        matched = [w for w in words if re.search(rf"\b{re.escape(w)}\b", text, re.I)]
        if matched:
            hits.append((page, matched))
    return hits

for label, words in [("all three words", ["assumptions", "linear", "regression"]),
                     ("just 'assumptions'", ["assumptions"])]:
    hits = keyword_search(words)
    pages = [p for p, _ in hits]
    found = sorted(RELEVANT & set(pages))
    prec = len(found) / len(pages) if pages else 0.0
    rec = len(found) / len(RELEVANT)
    print(f"searching for {label}:")
    for page, matched in hits:
        tag = "RELEVANT" if page in RELEVANT else "not relevant"
        print(f"    page {page:>3}  matched {str(matched):<32} {tag}")
    print(f"    precision {prec:.2f}   recall {rec:.2f}   missed {sorted(RELEVANT - set(pages))}\n")

Two different failures, and both are fatal.

Searching all three words drags in every page that says "assumptions" in an unrelated sense, so
**precision falls to 0.40**. Searching the key word alone **misses both relevant pages**, so
recall is **0.00** — because neither of them uses the word. Page 461 is about exactly this
topic and never says "assumptions" once.

No amount of cleverness with word matching fixes that. The word is not there.

## Part B — The retrieval pipeline

Three paragraphs, one per player. The question is "How many runs has Kohli scored?"

In [ ]:
docs = [
    "Virat Kohli is an Indian batter known for chasing targets. He has scored "
    "more than 26000 runs across formats and holds many records for centuries in run chases.",
    "Jasprit Bumrah is a fast bowler with an unusual action. He is known for "
    "yorkers at the death and has taken hundreds of wickets for India.",
    "Rohit Sharma is an opening batter famous for double centuries in one day "
    "internationals. He captains the side and is known for pulling short balls.",
]
names = ["Kohli", "Bumrah", "Rohit"]

vec = TfidfVectorizer(stop_words="english")
doc_vectors = vec.fit_transform(docs)
print(f"each paragraph is now a vector of {doc_vectors.shape[1]} numbers")

def retrieve(query):
    q = vec.transform([query])                 # the SAME vectoriser as the documents
    sims = cosine_similarity(q, doc_vectors)[0]
    for name, s in sorted(zip(names, sims), key=lambda x: -x[1]):
        print(f"  {name:<8} {s:.4f}  {'#' * int(s * 40)}")
    return sims

sims = retrieve("How many runs has Kohli scored?")
print(f"\nretrieved: {names[int(np.argmax(sims))]}")

That is the whole mechanism: text to vectors, compare, take the best. Load, split, embed,
store, retrieve. Swap in any embedding model and the shape is identical.

## Part C — Where it breaks

Now ask the same thing in different words. "Pursuing a total" means "chasing targets".

**Predict the three numbers before you run it.**

In [ ]:
sims = retrieve("Which player is best at pursuing a total?")

print(f"\nall similarities: {[round(float(s), 4) for s in sims]}")
if float(np.max(sims)) == 0.0:
    print(f"argmax says '{names[int(np.argmax(sims))]}' - but only because it takes the first")
    print("on a tie. Every score is EXACTLY zero, so there is no ranking at all.")

This is worse than retrieving the wrong paragraph. There is **no ranking**. The query shares no
content word with any paragraph, so the vectors are orthogonal and the system has nothing to go
on — it would hand the model whichever passage happened to come first.

**Be clear about what this does and does not show.** TF-IDF compares *words*. Real semantic
search needs an embedding model that places "chasing targets" and "pursuing a total" near each
other, and that model has to be trained on a lot of language. No embedding model is available
offline here, so this notebook can show the lexical version **failing** but cannot show the
semantic version succeeding. That failure is exactly why the system design has an embedding
model as its own component.

## Part D — Why retrieve at all

In [ ]:
words_per_page, pages, top_k = 500, 1000, 5
tokens_per_word = 1 / 0.75          # a common rule of thumb for English, not a measurement

whole     = pages * words_per_page * tokens_per_word
retrieved = top_k * words_per_page * tokens_per_word

print(f"{'sent to the model':<26} {'pages':>7} {'~tokens':>12}")
print(f"{'the whole book':<26} {pages:>7} {whole:>12,.0f}")
print(f"{'top-' + str(top_k) + ' pages':<26} {top_k:>7} {retrieved:>12,.0f}")
print(f"\n{whole/retrieved:.0f}x fewer tokens")

Two problems, both fixed by retrieving first. The whole book does not fit in a context window.
And if it did, you would pay for every one of those tokens to answer a single question.

## What to take away

- **Sketch any LLM application and three problems appear.** Understand and generate; run a
  model that big; join all the parts together.
- **The first is solved by using an LLM. The second by using an LLM API.** What is left is
  **orchestration**, and that is LangChain's job.
- **Keyword search fails two ways** — precision 0.40 with all three words, recall 0.00 with the
  key word alone.
- **Semantic search is embed, compare, take the top k**, with the query using the *same* model.
- **Word matching cannot see meaning** — the paraphrase scored exactly 0.0000 against
  everything.

**Exercises**

1. Add a page to `TEXTBOOK` that answers the question in completely different words. Confirm
   that keyword search never finds it, at any query you try.
2. Find a paraphrase of the Kohli question that TF-IDF *does* retrieve correctly. How much
   shared vocabulary did you have to leave in?
3. Change `top_k` and `words_per_page` in Part D to match a document set you actually have.
4. List the five components from the lesson's system design, and say which one you would
   struggle most to write yourself. That is the one LangChain saves you the most on.